# Generar voz a partir de una muestra propia

Este cuaderno prueba s?ntesis de voz condicionada por una grabaci?n breve. El audio de referencia y los resultados permanecen en el equipo. Usa solo tu propia voz o una voz para la que tengas autorizaci?n expl?cita.

```text
Texto + referencia de voz ? Chatterbox Multilingual ? WAV generado
```

La primera carga descarga los pesos del modelo y puede ocupar varios gigabytes.


## Preparar un entorno aislado

Requiere Python 3.11. Desde la ra?z del repositorio crea un entorno virtual, act?valo e instala `requirements-chatterbox.txt`. Selecciona ese entorno como kernel del cuaderno. En Windows la activaci?n se hace con `.venv-voz\Scripts\Activate.ps1`; en macOS/Linux, con `source .venv-voz/bin/activate`.


In [ ]:
!pip install -r requirements-chatterbox.txt


In [ ]:
import logging
import re
import time
import warnings
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, Markdown, display

warnings.filterwarnings("ignore")
logging.getLogger("chatterbox").setLevel(logging.ERROR)        # sin los avisos internos del modelo
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

SALIDAS = Path("audios")                 # aquí se guarda todo lo que generemos
SALIDAS.mkdir(exist_ok=True)

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("PyTorch", torch.__version__, "· se va a usar:", DISPOSITIVO)

## Cargar el modelo

Se inicializa Chatterbox Multilingual en el dispositivo disponible. La primera ejecuci?n descarga los archivos del modelo; el tiempo depende de la conexi?n y del hardware. La GPU acelera la generaci?n, pero no es imprescindible.


In [ ]:
from chatterbox.mtl_tts import SUPPORTED_LANGUAGES, ChatterboxMultilingualTTS

inicio = time.time()
modelo = ChatterboxMultilingualTTS.from_pretrained(device=DISPOSITIVO)
VOZ_DE_FABRICA = modelo.conds          # la voz que trae el modelo; la usaremos en el paso 8 para comparar

print(f"Modelo cargado en {time.time() - inicio:.0f} s · genera audio a {modelo.sr} Hz")
print(len(SUPPORTED_LANGUAGES), "idiomas:", ", ".join(f"{codigo} ({nombre})" for codigo, nombre in SUPPORTED_LANGUAGES.items()))

## Grabar o importar una referencia

Graba unos segundos en un lugar silencioso, habla con naturalidad y evita m?sica, reverberaci?n y otras voces. Tambi?n puedes cargar una grabaci?n tuya. El archivo `mi_voz.wav` es personal y no debe compartirse sin permiso.


In [ ]:
ARCHIVO_DE_VOZ = Path("mi_voz.wav")
SEGUNDOS = 15
REGRABAR = False
IMPORTAR = None                # por ejemplo "grabacion_del_movil.mp3"; None para grabar aquí

if IMPORTAR:
    audio, frecuencia = librosa.load(IMPORTAR, sr=None, mono=True)
    sf.write(ARCHIVO_DE_VOZ, audio, frecuencia)
    print(f"Importado {IMPORTAR} → {ARCHIVO_DE_VOZ}")

elif ARCHIVO_DE_VOZ.exists() and not REGRABAR:
    print(f"Ya existe {ARCHIVO_DE_VOZ}: se usa esa grabación. Para repetirla, pon REGRABAR = True.")

else:
    import sounddevice as sd

    microfono = sd.query_devices(kind="input")
    frecuencia = int(microfono["default_samplerate"])
    print("Micrófono:", microfono["name"])

    for cuenta in ["3", "2", "1"]:
        print(cuenta, end=" … ", flush=True)
        time.sleep(1)
    print("¡HABLA!")

    grabacion = sd.rec(int(SEGUNDOS * frecuencia), samplerate=frecuencia, channels=2, dtype="float32")
    for quedan in range(SEGUNDOS, 0, -1):
        print(quedan, end=" ", flush=True)
        time.sleep(1)
    sd.wait()

    sf.write(ARCHIVO_DE_VOZ, grabacion[:, 0], frecuencia)
    print(f"\nGuardado en {ARCHIVO_DE_VOZ}")

## Comprobar la grabaci?n

Escucha la muestra y revisa duraci?n, nivel y silencios. Si hay clipping, ruido o reverberaci?n marcada, repite la captura antes de generar audio.


In [ ]:
mi_voz, frecuencia = librosa.load(ARCHIVO_DE_VOZ, sr=None, mono=True)
duracion = len(mi_voz) / frecuencia
pico = float(np.abs(mi_voz).max())
volumen = float(np.sqrt((mi_voz ** 2).mean()))

fig, eje = plt.subplots(figsize=(9, 2.2))
eje.plot(np.arange(len(mi_voz)) / frecuencia, mi_voz, linewidth=0.4, color="#0147FF")
eje.set_ylim(-1, 1)
eje.set_xlabel("segundos")
eje.set_title("Tu grabación", loc="left", fontweight="bold")
eje.spines[["top", "right"]].set_visible(False)
plt.show()

display(Audio(mi_voz, rate=frecuencia))


def revisar(que, bien, consejo):
    print(f"✔ {que}" if bien else f"✘ {que}\n   {consejo}")


revisar(f"Duración: {duracion:.1f} s", duracion >= 8, "Graba al menos 10 segundos.")
revisar(f"Volumen medio: {volumen:.3f}", volumen >= 0.01, "Se oye muy bajo: acércate al micrófono o sube la entrada en los ajustes de sonido.")
revisar(f"Pico máximo: {pico:.2f}", pico < 0.99, "La grabación satura: aléjate un poco o baja la entrada del micrófono.")

Una referencia limpia suele dar una salida m?s estable. La siguiente secci?n re?ne funciones auxiliares para generar y guardar los resultados en `audios/`.


In [ ]:
ESTILOS = {
    "plano":     dict(exaggeration=0.25, cfg_weight=0.5),
    "natural":   dict(exaggeration=0.5, cfg_weight=0.5),      # los valores por defecto del modelo
    "expresivo": dict(exaggeration=0.7, cfg_weight=0.3),
    "teatral":   dict(exaggeration=0.9, cfg_weight=0.3),
}


def hablar(texto, estilo="natural", idioma="es", semilla=7, guardar=None, intentos=3, **ajustes):
    """Devuelve el audio del texto leído con tu voz. Con `guardar`, además lo escribe en la carpeta de audios."""
    opciones = {**ESTILOS[estilo], **ajustes}
    limite = 2.0 + 0.075 * len(texto)         # segundos que, como mucho, debería durar este texto
    for intento in range(intentos):
        torch.manual_seed(semilla + intento)
        onda = modelo.generate(texto, language_id=idioma, audio_prompt_path=str(ARCHIVO_DE_VOZ), **opciones)
        audio = onda.squeeze(0).cpu().numpy()
        if len(audio) / modelo.sr <= limite or idioma in {"zh", "ja", "ko"}:
            break
        print(f"   ↻ «{texto[:40]}» ha salido demasiado largo ({len(audio) / modelo.sr:.1f} s): se repite con otra semilla")
    if guardar:
        sf.write(SALIDAS / guardar, audio, modelo.sr)
    return audio


def escuchar(audio, titulo=None):
    if titulo:
        display(Markdown(f"**{titulo}** · {len(audio) / modelo.sr:.1f} s"))
    display(Audio(audio, rate=modelo.sr))

In [ ]:
inicio = time.time()
primera = hablar("Hola. Esta frase no la he dicho yo: la ha generado un modelo a partir de quince segundos de mi voz.",
                 guardar="01_primera_frase.wav")

print(f"{len(primera) / modelo.sr:.1f} s de audio generados en {time.time() - inicio:.0f} s")
escuchar(primera, "Tu voz clonada")
escuchar(mi_voz[: int(8 * frecuencia)], "Tu grabación original, para comparar")

## Ajustar la expresividad

Los par?metros de cada estilo modifican la intensidad expresiva y el ajuste al texto. Compara varias salidas con la misma frase; una semilla fija facilita la comparaci?n, aunque no garantiza resultados id?nticos entre versiones del modelo.


In [ ]:
frase = "¡No me lo puedo creer! Llevo toda la semana preparando esta clase, y ahora resulta que la voz que estáis oyendo no la he grabado yo."

for estilo, ajustes in ESTILOS.items():
    audio = hablar(frase, estilo=estilo, guardar=f"02_estilo_{estilo}.wav")
    escuchar(audio, f"{estilo} · exaggeration {ajustes['exaggeration']} · cfg_weight {ajustes['cfg_weight']}")

## El texto afecta a la lectura

La puntuaci?n y la segmentaci?n orientan las pausas y la entonaci?n. Prueba cambios peque?os y escucha el resultado completo antes de conservarlo.


In [ ]:
para_leer = "El modelo Chatterbox (500M de parámetros) clona una voz con 10-20 s de audio y soporta 23 idiomas."
para_oir = "Chatterbox es un modelo pequeño: quinientos millones de parámetros. ¿Y qué necesita para clonar una voz? Quince segundos de audio. Nada más."

escuchar(hablar(para_leer, estilo="expresivo", guardar="03_escrito_para_leer.wav"), "Escrito para leer")
escuchar(hablar(para_oir, estilo="expresivo", guardar="03_escrito_para_oir.wav"), "Escrito para oír")

Prueba una frase propia con dos estilos distintos y compara claridad, ritmo y parecido. No uses nombres ni datos personales de terceros en los ejemplos.


In [ ]:
# Prueba tú
mi_frase = "Que ganas tengo de comerme un plato de croquetas"
escuchar(hablar(mi_frase, estilo="expresivo"), "Mi frase")

## Generar un texto largo

El siguiente ejemplo divide un guion en segmentos, sintetiza cada parte y une el audio con pausas breves. Ajusta el texto y revisa la uni?n entre frases.


In [ ]:
GUION = [
    ("Hace un mes, yo no sabía nada de esto.", "natural"),
    ("Hoy he grabado quince segundos de mi voz con el micrófono del portátil, y un modelo de código abierto ha aprendido a imitarla.", "natural"),
    ("¿Asusta un poco? Sí, bastante.", "expresivo"),
    ("Pero también es una herramienta extraordinaria: para quien ha perdido la voz, para doblar un vídeo a otro idioma, para que un texto largo se pueda escuchar.", "expresivo"),
    ("Lo importante es saber cómo funciona, qué puede hacer… y qué no.", "plano"),
    ("¡Y eso es justo lo que vamos a ver hoy, con vuestra propia voz!", "teatral"),
]


def leer_guion(guion, pausa=0.35, guardar=None):
    """Genera cada frase con su estilo y las une, dejando `pausa` segundos de silencio entre una y otra."""
    silencio = np.zeros(int(pausa * modelo.sr), dtype="float32")
    trozos = []
    for n, (texto, estilo) in enumerate(guion, start=1):
        print(f"{n}/{len(guion)} [{estilo}] {texto[:70]}")
        trozos += [hablar(texto, estilo=estilo), silencio]
    audio = np.concatenate(trozos[:-1])
    if guardar:
        sf.write(SALIDAS / guardar, audio, modelo.sr)
    return audio


inicio = time.time()
lectura = leer_guion(GUION, guardar="04_lectura_con_guion.wav")
print(f"\n{len(lectura) / modelo.sr:.0f} s de audio generados en {time.time() - inicio:.0f} s")
escuchar(lectura, "Lectura con guion")

La funci?n de lectura reutiliza el mismo estilo en cada segmento y guarda el resultado en `audios/`. Los WAV generados son archivos locales e ignorados por Git.


In [ ]:
def leer(texto, estilo="expresivo", pausa=0.35, guardar=None):
    """Parte un texto en frases y lo lee entero con el mismo estilo."""
    frases = []
    for frase in re.split(r"(?<=[.!?…])\s+", " ".join(texto.split())):
        if frases and len(frases[-1]) < 40:          # una frase muy corta se une a la siguiente
            frases[-1] += " " + frase
        else:
            frases.append(frase)
    if len(frases) > 1 and len(frases[-1]) < 40:     # y si la corta es la última, a la anterior
        frases[-2:] = [frases[-2] + " " + frases[-1]]
    return leer_guion([(frase, estilo) for frase in frases], pausa=pausa, guardar=guardar)


TEXTO = """
Buenos días a todos. Antes de empezar, una advertencia: esta bienvenida no la estoy leyendo yo.
La escribí anoche, y la voz que oís la ha puesto un modelo. ¿Se nota? Decídmelo vosotros al terminar.
"""

escuchar(leer(TEXTO, guardar="05_mi_texto.wav"), "Mi texto")

## Probar otro idioma

El modelo admite varios idiomas. La calidad y la pronunciaci?n var?an seg?n el idioma; revisa cuidadosamente el audio, especialmente nombres propios y traducciones.


In [ ]:
OTROS_IDIOMAS = {
    "en": "Hello! This is my own voice, speaking a language I never recorded.",
    "fr": "Bonjour ! C'est ma propre voix, dans une langue que je n'ai jamais enregistrée.",
    "it": "Ciao! Questa è la mia voce, in una lingua che non ho mai registrato.",
}

for idioma, texto in OTROS_IDIOMAS.items():
    audio = hablar(texto, idioma=idioma, cfg_weight=0.0, guardar=f"06_idioma_{idioma}.wav")
    escuchar(audio, SUPPORTED_LANGUAGES[idioma])

Compara una frase breve con distintos ajustes y verifica que el mensaje siga siendo comprensible. El audio sint?tico debe presentarse como tal.


## Comparar la identidad de voz

La m?trica del ejemplo compara representaciones de hablante para orientar una escucha cr?tica. No es una prueba de identidad ni sustituye el consentimiento.


In [ ]:
def huella(audio, frecuencia_audio):
    """Vector que resume quién habla en un audio, calculado con el codificador de voz del modelo."""
    audio_16k = librosa.resample(np.asarray(audio, dtype="float32"), orig_sr=frecuencia_audio, target_sr=16_000)
    vector = np.asarray(modelo.ve.embeds_from_wavs([audio_16k], sample_rate=16_000)).reshape(-1)
    return vector / np.linalg.norm(vector)


tu_voz_en_el_modelo = modelo.conds          # el modelo recuerda la última voz que se le dio: la apartamos un momento
modelo.conds = VOZ_DE_FABRICA
torch.manual_seed(7)
onda = modelo.generate("Y esta es la voz que trae el modelo cuando no se le da ninguna grabación.", language_id="es")
modelo.conds = tu_voz_en_el_modelo
voz_de_fabrica = onda.squeeze(0).cpu().numpy()
escuchar(voz_de_fabrica, "La voz de fábrica del modelo")

referencia = huella(mi_voz, frecuencia)
print("Parecido con tu grabación (1 = la misma voz)\n")
for nombre, audio in [("Tu primera frase clonada", primera), ("La lectura con guion", lectura), ("La voz de fábrica del modelo", voz_de_fabrica)]:
    print(f"  {nombre:32} {float(referencia @ huella(audio, modelo.sr)):.2f}")

Una puntuaci?n m?s alta indica mayor proximidad seg?n ese codificador, no que el audio sea una reproducci?n perfecta. Tambi?n hay que valorar pronunciaci?n, artefactos y contexto.


In [ ]:
import perth

detector = perth.PerthImplicitWatermarker()

clon_guardado, frecuencia_clon = librosa.load(SALIDAS / "01_primera_frase.wav", sr=None)

print("Marca de agua (1 = generado por el modelo, 0 = no)\n")
print("  Tu grabación original :", detector.get_watermark(mi_voz, sample_rate=frecuencia))
print("  La voz clonada        :", detector.get_watermark(clon_guardado, sample_rate=frecuencia_clon))

## Transparencia y uso responsable

Usa ?nicamente muestras propias o autorizadas. No suplantes a otra persona ni presentes audio generado como una grabaci?n aut?ntica. Conserva la marca de agua disponible y comunica cu?ndo el contenido es sint?tico.

El cuaderno utiliza Chatterbox Multilingual y Perth, de Resemble AI, conforme a las licencias y condiciones de sus proyectos. Consulta sus repositorios antes de redistribuir modelos o audio.
